In [ ]:
%matplotlib inline
import time, os, sys
import math
import numpy as np
import matplotlib.pyplot as plt
from joblib import Memory
from sklearn.model_selection import train_test_split
from pathlib import Path
from functools import partial
import ml_visualization
import ml_utilities
import pandas as pd
import tensorflow as tf
import random
from tensorflow.python.client import device_lib
import os

np.random.seed(42)
random.seed = 42

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Nasconde messaggi di debug
os.environ["CUDA_VISIBLE_DEVICES"] = '1'  # Rende visibile solo la GPU 0
print('Tensorflow Version', tf.__version__)
print ("\nHardware Devices:")
print (device_lib.list_local_devices())

In [ ]:
db_name = 'ml_multi' #'ml_bin' 'ml_multi'

if db_name == 'ml_bin':
    db_path = 'DBs/CaniGatti_ML18_Es8_2020'
    train_filelist = 'BinaryTrainingSet.txt'
    test_filelist = 'BinaryTestSet.txt'
    label_filelist = db_path + '/BinaryLabels.txt'
    class_num = 2   # caso due classi
    validation_size = 200
    test_has_labels = True
elif db_name == 'ml_multi':
    db_path = 'DBs/CaniGatti_ML18_Es8_2020'
    train_filelist = 'MulticlassTrainingSet.txt'
    test_filelist = 'MulticlassTestSet.txt'
    label_filelist = db_path + '/MulticlassLabels.txt'
    class_num = 12   # caso 12 classi - singoli animali
    validation_size = 200
    test_has_labels = True
else:
    raise ValueError('Database non valido.')

checkpoints_path = './saved_models'
exp_path = './exp_cache'

# Predisposizione di un'area di caching su disco che velocizza la ri-esecuzione di chiamate di funzioni con gli stessi parametri
memory = Memory(exp_path, verbose=0)

# Caricamento delle immagini
print('Caricamento in corso ...')
start = time.time()
raw_images_x, label_y = ml_utilities.load_labeled_dataset(train_filelist, db_path, cache=memory)

# Carica le etichette di classe
label_names = [line.rstrip('\n') for line in open(label_filelist,'r')]

print('Caricate %d immagini in %.2f s.' % (len(raw_images_x), time.time() - start))
for i in range(len(label_names)):
    print('{}: {}'.format(label_names[i], np.count_nonzero(label_y == i)))

In [ ]:
mobilenet_version = 'small'  # 'small' 'default'

if mobilenet_version == 'default':
    image_side = 224
    width_multiplier = 1.0
elif mobilenet_version == 'small':
    image_side = 128
    width_multiplier = 0.5
else:
    raise ValueError('Versione di MobileNet non supportata.')

# Definizione della shape dei tensori delle immagini
IMG_SHAPE = (image_side, image_side, 3)

print('Resizing in corso ...')
start = time.time()
resized_image_x = ml_utilities.resize_images(raw_images_x, image_side, image_side, cache=memory)
print('Resizing completato in %.2f s.' % (time.time() - start))

In [ ]:
def compute_accuracy(y_true, y_pred):
    correct_predictions = 0
    # iterate over each label and check
    for true, predicted in zip(y_true, y_pred):
        if true == predicted:
            correct_predictions += 1
    # compute the accuracy
    accuracy = correct_predictions/len(y_true)
    return accuracy

In [ ]:
from sklearn.model_selection import KFold, StratifiedKFold
import pandas as pd
from keras.preprocessing.image import ImageDataGenerator
from skimage import io
datagen = ImageDataGenerator(
            rotation_range=45,
            width_shift_range=0.2,
            height_shift_range=0.2,
            shear_range=0.2,
            zoom_range=0.2,
            horizontal_flip=True,
            fill_mode='nearest')

folds = KFold(5)
df = pd.DataFrame()
best_accuracy = None
minibatch_size_test = 50

raw_test_images_x, test_y = ml_utilities.load_labeled_dataset(test_filelist, db_path, cache=memory)
resized_test_image_x = ml_utilities.resize_images(raw_test_images_x, image_side, image_side, cache=memory)

test_dataset = tf.data.Dataset.from_tensor_slices((resized_test_image_x, test_y))
test_dataset = test_dataset.batch(minibatch_size_test, drop_remainder=False)

train_generator = datagen.flow(x=resized_image_x, y=label_y,batch_size=1000)

shapes_added, labels_added = train_generator.next()
resized_image_x = np.concatenate((resized_image_x, shapes_added),axis=0)
label_y = np.concatenate((label_y, labels_added),axis=0)

for i, (train_idx, val_idx) in enumerate(folds.split(resized_image_x, label_y)):


    # Creazione della rete
    base_model = tf.keras.applications.MobileNet(input_shape=IMG_SHAPE,
                                                 alpha=width_multiplier,
                                                 include_top=False,
                                                 pooling='avg',
                                                 weights='imagenet')

    # Se trainable è uguale a False i pesi della rete sono congelati
    # In questo caso vogliamo permettere l'addestramento dei pesi
    base_model.trainable = True

    # Definisce la loss
    loss_function = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

    # Definisce l'ottimizzatore
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)

    # Le metriche offerte da Keras accumulano i valori su più batch
    # Loss media sul training set
    train_loss = tf.keras.metrics.Mean(name='train_loss')
    # Accuratezza sul training set
    train_accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')

    # Loss media sul validation/test set
    val_loss = tf.keras.metrics.Mean(name='val_loss')
    # Accuratezza sul validation/test set
    val_accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')

    # Adatta la rete usando Functional API di Keras
    inputs = tf.keras.Input(shape=IMG_SHAPE)
    feature = base_model(inputs)
    dense_layer = tf.keras.layers.Dense(class_num)
    logits = dense_layer(feature)
    softmax_layer = tf.keras.layers.Softmax()
    probs = softmax_layer(logits)
    model = tf.keras.Model(inputs=inputs, outputs=[logits, probs])

    # NOTA: non viene chiamata la funzione .compile di Keras in
    # quanto eseguiamo il training a più basso livello senza
    # utilizzare la funzione .fit

    # Funzione di train su un minibatch (iterazione)
    @tf.function
    def train_step(x, y):
        with tf.GradientTape() as tape:
            logits, _ = model(x, training=True)
            loss_value = loss_function(y, logits)
        gradients = tape.gradient(loss_value, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        # Le metriche accumulano i valori (fino a chiamata di reset)
        train_loss(loss_value)
        train_accuracy_metric.update_state(y,logits)

    # Funzione di inference su un minibatch
    @tf.function
    def test_step(x, y):
        logits, probs = model(x, training=False)
        v_loss = loss_function(y, logits)
        # Le metriche accumulano i valori (fino a chiamata di reset)
        val_loss(v_loss)
        val_accuracy_metric.update_state(y,logits)
        return probs


    train_x = resized_image_x[train_idx]
    train_y = label_y[train_idx]
    validation_x = resized_image_x[val_idx]
    validation_y = label_y[val_idx]

    minibatch_size = 50

    training_set = tf.data.Dataset.from_tensor_slices((train_x, train_y))
    validation_set = tf.data.Dataset.from_tensor_slices((validation_x, validation_y))

    train_dataset = tf.data.Dataset.shuffle(training_set,buffer_size=1024,seed=1234).batch(minibatch_size)
    valid_dataset = tf.data.Dataset.shuffle(validation_set,buffer_size=1024,seed=1234).batch(minibatch_size)

    # Attenzione: eseguire questa cella più volte significa riprendere
    # l'addestramento da dove era stato lasciato, cioè non ricomincia
    # ogni volta dai pesi iniziali!
    n_epochs = 22

    epochs_training_loss = []
    epochs_validation_accuracy = []
    epochs_training_accuracy = []
    best_weights = []

    t_start = time.time()
    print("Start Training")

    total_train_patterns = 0

    for epoch in range(n_epochs):

        train_loss.reset_states()
        train_accuracy_metric.reset_states()
        val_loss.reset_states()
        val_accuracy_metric.reset_states()

        print("Epoch", epoch + 1, " ", end="")


        # Train su tutti i minibatch dell'epoca
        for X_minibatch, y_minibatch in train_dataset:
            total_train_patterns += len(X_minibatch)
            print(".", end="", flush = True)
            train_step(X_minibatch, y_minibatch)

        # Evaluation
        for X_minibatch, y_minibatch in valid_dataset:
            print("+", end="", flush = True)
            test_step(X_minibatch, y_minibatch)

        epochs_training_loss.append(train_loss.result().numpy())
        epochs_training_accuracy.append(train_accuracy_metric.result().numpy() * 100)
        epochs_validation_accuracy.append(val_accuracy_metric.result().numpy() * 100)

        print("  Train loss: %.4f  Train acc: %.2f %%  Validation acc: %.2f %%" % (train_loss.result(), train_accuracy_metric.result() * 100, val_accuracy_metric.result() * 100))

        first_minibatch = True
        for X_minibatch, y_minibatch in test_dataset:
            minibatch_probs = test_step(X_minibatch, y_minibatch)
            if first_minibatch:
                test_probs = minibatch_probs.numpy()
                first_minibatch = False
            else:
                test_probs = np.concatenate((test_probs, minibatch_probs))
        predicted_y = test_probs.argmax(axis=1)
        acc = compute_accuracy(test_y,predicted_y)
        print(f"acc {acc}")

    print(" %.2f %%" % (100 * val_accuracy_metric.result()))
    t_elapsed = time.time()-t_start
    print (' -> %d patterns (%.2f sec.) -> %.2f patt/sec' % (len(test_y), t_elapsed, len(test_y) / t_elapsed))
    predicted_y = test_probs.argmax(axis=1)

    print('Salvataggio del modello...')
    # Salva tutto: pesi, ottimizzatore, ecc.
    save_path = str(Path(checkpoints_path) / ("dogcat_model_%s_with_%d_classes_%d.h5" % (db_name, class_num,i)))

    # .save() può restituire dei warning relativi alla deprecazione di alcune componenti
    # interne di TensorFlow. Si tratta di un bug già sistemato nelle versioni nightly
    # e che scomparirà nelle prossime versioni.
    model.save(save_path)

    t_elapsed = time.time()-t_start
    print (' -> %d patterns (%.2f sec.) -> %.2f patt/sec' % (total_train_patterns, t_elapsed, total_train_patterns / t_elapsed))

    ml_visualization.plot_performance_curves(epochs_training_loss, epochs_training_accuracy, epochs_validation_accuracy)

    if not test_has_labels:
        # Non sono fornite etichette per il problema di classificazione sui singoli animali
        raise ValueError('Il test set per il problema scelto non ha etichette di classe')

    # Carica il modello salvato in precedenza su disco.
    # Attenzione: se questa istruzione restituisce un errore
    # è possibile commentarla e utilizzare il modello
    # già memorizzato nella variabile "model". In alternativa
    # è possibile aggiornare a TensorFlow 2.2.0 o superiore.
    model = tf.keras.models.load_model(save_path)

    print("Computing Accuracy on the Test Set ", end="" )
    t_start = time.time()

    # Riutilizziamo gli oggetti metrics usati per il validation set
    val_loss.reset_states()
    val_accuracy_metric.reset_states()

    first_minibatch = True
    for X_minibatch, y_minibatch in test_dataset:
        minibatch_probs = test_step(X_minibatch, y_minibatch)
        if first_minibatch:
            test_probs = minibatch_probs.numpy()
            first_minibatch = False
        else:
            test_probs = np.concatenate((test_probs, minibatch_probs))

    print(" %.2f %%" % (100 * val_accuracy_metric.result()))
    t_elapsed = time.time()-t_start
    print (' -> %d patterns (%.2f sec.) -> %.2f patt/sec' % (len(test_y), t_elapsed, len(test_y) / t_elapsed))
    predicted_y = test_probs.argmax(axis=1)
    df = pd.concat((df, pd.DataFrame(predicted_y)), axis=1)
    print(predicted_y.shape, test_y.shape)
    ml_visualization.plot_confusion_matrix(test_y, predicted_y, label_names, figsize=(10,8))

df.to_pickle("predictions.pkl")

In [ ]:
df_label = df
df_label = df_label.mode(axis=1)
label_y = df_label.mean(axis=1).round()
np.float32(label_y)
print(f"accuracy {compute_accuracy(test_y,label_y)}")
ml_visualization.plot_confusion_matrix(test_y, label_y, label_names, figsize=(10,8))